# NB164 — CKM from the Covering Tower

## The CKM Mechanism in the Solenoid

The CKM matrix comes from the covering tower structure, with two distinct contributions:

**1. V_cb ≈ 0.04 from the tower (this notebook):**
- The cascade creates non-constant j₃ profiles on the p=7 fiber (the Higgs VEV, NB53)
- Different a5 sectors (up vs down) have different profiles (CRT isospin step Δci=-84)
- The NB55 two-level tower (C₆ + C₄₂) couples fiber to base through Pi/√7
- The C₄₂ cycle Laplacian connects gen0↔gen2 at fiber boundaries (7 connections)
- Different fiber VEV profiles → different eigenvectors → V_cb from misalignment
- At t_hop = √κ = 1/P₄^{1/4} (geometric mean coupling), V_cb = 0.040

**2. V_us ≈ 0.22 from F-N mass texture (NB163):**
- sin θ_C = √(m_d/m_s) = 1/√20 = 0.224 (from cascade mass ratio)
- The tower gives V_us ≈ 0.004 (second-order: gen1↔gen2 requires two hops)
- The dominant Cabibbo mixing comes from the mass texture, not the tower

This matches the SM hierarchy: V_us ∝ λ, V_cb ∝ λ² where λ = sin θ_C.

## Key Results
| CKM element | Tower | F-N | Combined | PDG |
|-------------|-------|-----|----------|-----|
| V_us | 0.004 | 0.224 | 0.224 | 0.225 |
| V_cb | 0.040 | — | 0.040 | 0.041 |
| V_ub | 0.016 | — | 0.016 | 0.004 |

V_ub is off by 4×. This may need the 3-level tower or additional physics.

In [2]:
# S0 — Complete CKM computation with cascade fiber VEV on NB55 tower
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod
from solenoid_system import SolenoidSystem
from solenoid_algebra import SA

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
P3 = p1*p2*p3
kappa = 1.0 / np.sqrt(P4)
omega = 2*np.pi

# Cascade integration
sys0 = SolenoidSystem(primes=primes)
branches = sys0.all_branches()
cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)
from solenoid_jax import warmup
warmup()
res = sys0.integrate_all_branches(branches, cis.astype(float), float(P4+1), backend='jax')
j3_vals = np.array([br[3] for br in branches])

def get_R3_profile(ci_val):
    idx = np.where(cis == ci_val)[0][0]
    R3 = np.array([res[br][idx, 3] for br in branches])
    R3_w = np.mod(R3, 2*np.pi)
    R3_w[R3_w > np.pi] -= 2*np.pi
    return np.array([np.mean(R3_w[j3_vals == j]) for j in range(7)])

# NB55 two-level tower (C_6 + C_42 = 48 sites)
N = 48
def cycle_lap(n):
    L = np.zeros((n, n))
    for j in range(n):
        L[j,j] = 2; L[j,(j+1)%n] = -1; L[j,(j-1)%n] = -1
    return L
Pi = np.zeros((6, 42))
for j in range(42): Pi[j%6, j] = 1
def build_kin(t_hop):
    H = np.zeros((N,N))
    H[:6,:6] = cycle_lap(6)
    H[6:,6:] = cycle_lap(42)
    H[:6,6:] = t_hop * Pi / np.sqrt(7)
    H[6:,:6] = t_hop * (Pi / np.sqrt(7)).T
    return H
gen_labels = np.zeros(N, dtype=int)
for j in range(6): gen_labels[j] = j%3
for j in range(42): gen_labels[6+j] = (j%6)%3

def build_mass(ci_val, t_hop, lam=1.0):
    phi = get_R3_profile(ci_val)
    H = build_kin(t_hop)
    avg = np.mean(np.abs(phi))
    for j in range(6): H[j,j] += 3*lam*avg**2
    for j in range(42): H[6+j,6+j] += 3*lam*phi[j//6]**2
    return H

def compute_ckm(ci_d, ci_u, t_hop):
    _, U_d = np.linalg.eigh(build_mass(ci_d, t_hop))
    _, U_u = np.linalg.eigh(build_mass(ci_u, t_hop))
    def gdom(ev):
        s={}; u=set()
        for g in range(3):
            bk,bw=None,0
            for k in range(N):
                if k in u: continue
                w=np.sum(ev[gen_labels==g,k]**2)
                if w>bw: bw=w; bk=k
            s[g]=(bk,bw); u.add(bk)
        return s
    gd,gu = gdom(U_d), gdom(U_u)
    V=np.zeros((3,3))
    for i in range(3):
        for j in range(3):
            V[i,j]=abs(np.dot(U_d[:,gd[i][0]], U_u[:,gu[j][0]]))
    return V, max(gd[g][1] for g in range(3))

# ═══ CASCADE FIBER PROFILES ═══
print('=== Cascade fiber profiles at wrapping crossings ===')
for ci, lab in [(11,'DOWN gen2 a5=0'), (17,'UP-1 gen2 a5=1')]:
    phi = get_R3_profile(ci)
    fft = np.fft.fft(phi)
    print(f'{lab} (ci={ci}): Fourier m0={abs(fft[0])/7:.3f} m1={abs(fft[1])/7:.3f} m2={abs(fft[2])/7:.3f} m3={abs(fft[3])/7:.3f}')

# ═══ FINE t_hop SWEEP ═══
print(f'\n=== V_cb vs t_hop (fine sweep, lam=1) ===')
print(f'{"t_hop":>8} {"V_us":>8} {"V_cb":>8} {"V_ub":>8}')
t_vals = np.arange(0.20, 0.40, 0.01)
vcb_vals = []
for t in t_vals:
    V,_ = compute_ckm(11, 17, t)
    vcb_vals.append(V[1,2])
    print(f'{t:8.3f} {V[0,1]:8.4f} {V[1,2]:8.4f} {V[0,2]:8.4f}')

# Interpolate to find t_hop where V_cb = 0.041
from scipy.interpolate import interp1d
f_vcb = interp1d(np.array(vcb_vals), t_vals)
t_match = float(f_vcb(0.041))
print(f'\nV_cb = 0.041 (PDG) at t_hop = {t_match:.4f}')

# ═══ NATURAL t_hop CANDIDATES ═══
print(f'\n=== Natural t_hop candidates ===')
H3 = P3/np.sqrt(P3**2 + omega**2*P4)  # cascade filter gain
candidates = [
    (np.sqrt(kappa),         f'sqrt(kappa) = P4^(-1/4) = {np.sqrt(kappa):.4f}'),
    (P3/(p4*np.sqrt(P4)),    f'P3/(p4*sqrt(P4)) = kappa*P3/p4 = {P3/(p4*np.sqrt(P4)):.4f}'),
    (H3,                     f'|H3| = cascade filter gain = {H3:.4f}'),
    (1/np.sqrt(p4),          f'1/sqrt(p4) = {1/np.sqrt(p4):.4f}'),
    (t_match,                f'V_cb=0.041 match = {t_match:.4f}'),
]
for val, desc in candidates:
    V,mw = compute_ckm(11, 17, val)
    print(f'  {desc}: V_cb={V[1,2]:.4f} V_us={V[0,1]:.4f} V_ub={V[0,2]:.4f} gen_wt={mw:.3f}')

# ═══ UNDERSTANDING THE COUPLING STRUCTURE ═══
print(f'\n=== Generation coupling structure ===')
print(f'C_42 fiber boundaries: ALL 7 connect gen2↔gen0 (explains V_cb)')
print(f'C_6 Laplacian: connects gen0↔gen1, gen1↔gen2, gen2↔gen0 (all pairs)')
print(f'Gen1↔gen2 (Cabibbo) is second-order on the tower → V_us ≈ 0')
print(f'V_us comes from F-N mass texture: sin theta_C = 1/sqrt(20) = {1/np.sqrt(20):.4f}')

# ═══ SUMMARY ═══
V_best,_ = compute_ckm(11, 17, np.sqrt(kappa))
print(f'\n{"="*60}')
print(f'SUMMARY: CKM from the solenoid')
print(f'{"="*60}')
print(f'')
print(f'At t_hop = sqrt(kappa) = 1/P4^(1/4) = {np.sqrt(kappa):.4f}:')
print(f'  |V_us| = {V_best[0,1]:.4f}  (PDG: 0.225, from F-N: 0.224)')
print(f'  |V_cb| = {V_best[1,2]:.4f}  (PDG: 0.041)')
print(f'  |V_ub| = {V_best[0,2]:.4f}  (PDG: 0.004)')
print(f'')
print(f'Two mechanisms:')
print(f'  V_cb ~ 0.04: tower effect (cascade fiber VEV + inter-level coupling)')
print(f'  V_us ~ 0.22: F-N mass texture (cascade mass ratio sin theta_C = 1/sqrt(20))')
print(f'')
print(f'The tower parameter t_hop = sqrt(kappa) = geometric mean of cascade')
print(f'coupling (kappa) and kinetic scale (1). Natural coupling for the')
print(f'covering tower: sqrt(damping × kinetic) = sqrt(1/sqrt(P4)).')
print(f'')
print(f'OPEN: V_ub is 4x too large. May need 3-level tower or V_ub = V_us * V_cb.')

  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 1.67s
=== Cascade fiber profiles at wrapping crossings ===
DOWN gen2 a5=0 (ci=11): Fourier m0=0.179 m1=0.480 m2=0.178 m3=0.740
UP-1 gen2 a5=1 (ci=17): Fourier m0=0.157 m1=0.174 m2=0.894 m3=0.562

=== V_cb vs t_hop (fine sweep, lam=1) ===
   t_hop     V_us     V_cb     V_ub
   0.200   0.0031   0.0313   0.0155
   0.210   0.0032   0.0327   0.0155
   0.220   0.0034   0.0340   0.0155
   0.230   0.0035   0.0354   0.0155
   0.240   0.0037   0.0367   0.0156
   0.250   0.0038   0.0380   0.0156
   0.260   0.0039   0.0393   0.0156
   0.270   0.0041   0.0406   0.0156
   0.280   0.0042   0.0418   0.0156
   0.290   0.0044   0.0430   0.0156
   0.300   0.0045   0.0443   0.0156
   0.310   0.0046   0.0455   0.0156
   0.320   0.0048   0.0467   0.0156
   0.330   0.0049   0.0478   0.0156
   0.340   0.0050   0.0490   0.0157
   0.350   0.0051   0.0502   0.0157
   0.360   0.0053   0.0513   0.0157
   0.370   0.0054   0.0524   0.0157
   0.380   0.0